# 10. Cryptographic Verifiable AI: ZKML, Freivalds' Algorithm, and Homomorphic Encryption (CKKS)

- **Category:** 11_ai_safety_privacy_and_verification
- **Lineage & Citations:** Freivalds (1977 *Probabilistic Machines Can Verify Faster than Prove*); Gilad-Bachrach et al. (2016 *CryptoNets: Applying Neural Networks to Encrypted Data with High Throughput and Accuracy*); Cheon et al. (2017 *Homomorphic Encryption for Arithmetic of Approximate Numbers*); Kang et al. (2022/2024 *Scaling up Zero-Knowledge Machine Learning*).
- **Purpose:** Formulate and evaluate cryptographic verifiable inference and privacy-preserving execution: Freivalds' $O(n^2)$ randomized matrix multiplication verification ($A(Br) \stackrel{?}{=} Cr$), CKKS degree-2 and degree-4 polynomial activation approximations, and ZKML fixed-point integer arithmetic circuits.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [09. OOD Detection Abstention Conformal and Algorithmic Fairness](09_ood_detection_abstention_conformal_and_algorithmic_fairness.ipynb) $\to$ `🔗 Downstream Syntheses:` [01. Torch Compile Modes Backends and Graph Breaks](../12_compilation_and_performance/01_torch_compile_modes_backends_and_graph_breaks.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not attempt to evaluate standard non-polynomial ReLUs or Softmax inside CKKS homomorphic encryption without polynomial approximation (comparison cannot be computed homomorphically); do not exceed the ciphertext multiplicative depth budget without rescaling.

## Mathematical Foundations & Governing Equations

1. **Freivalds' Verification Residual**: $v = A(Br) - Cr \in \mathbb{R}^M, \quad r \in \{0, 1\}^N \implies \mathbb{P}(v = 0 \mid AB \ne C) \le \frac{1}{2}$
2. **CKKS Quadratic Activation**: $y = a x^2 + b x + c$
3. **CKKS Quartic Activation**: $y = c_4 x^4 + c_2 x^2 + c_0$
4. **ZKML Fixed-Point Projection**: $y = \frac{1}{S} \text{round}\left((x W + b) \cdot S\right)$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/11_ai_safety_privacy_and_verification/10_cryptographic_verifiable_ai_zkml_freivalds_and_homomorphic_ckks.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def freivalds_verify_residual(a, b, c, r):
    return a @ (b @ r) - c @ r

def ckks_degree2_activation(x, a, b, c):
    return float(a) * (x ** 2) + float(b) * x + float(c)

def ckks_degree4_activation(x, c4, c2, c0):
    x2 = x ** 2
    return float(c4) * (x2 ** 2) + float(c2) * x2 + float(c0)

def zkml_fixed_point_linear(x, weight, bias, scale):
    y = x @ weight + bias
    return torch.round(y * float(scale)) / float(scale)

In [ ]:
# Parity verification against golden reference tensors
v_valid = freivalds_verify_residual(
    weights["mat_a"], weights["mat_b"], weights["mat_c"], weights["rand_r"]
)
assert torch.max(torch.abs(v_valid - weights["freivalds_valid_ref"])) < 1e-5

v_tamp = freivalds_verify_residual(
    weights["mat_a"], weights["mat_b"], weights["mat_c_tampered"], weights["rand_r"]
)
assert torch.max(torch.abs(v_tamp - weights["freivalds_tampered_ref"])) < 1e-5

p2 = ckks_degree2_activation(weights["x_poly"], a=0.125, b=0.5, c=0.25)
assert torch.max(torch.abs(p2 - weights["poly2_out_ref"])) < 1e-5

p4 = ckks_degree4_activation(weights["x_poly"], c4=-0.005, c2=0.2, c0=0.15)
assert torch.max(torch.abs(p4 - weights["poly4_out_ref"])) < 1e-5

zk_out = zkml_fixed_point_linear(weights["zk_x"], weights["zk_w"], weights["zk_b"], scale=16.0)
assert torch.max(torch.abs(zk_out - weights["zk_out_ref"])) < 1e-5
print("All Freivalds, CKKS Polynomial, and ZKML parity assertions passed!")

In [ ]:
# Benchmarking cryptographic verification and homomorphic operators
t0 = time.perf_counter()
for _ in range(500):
    _ = freivalds_verify_residual(weights["mat_a"], weights["mat_b"], weights["mat_c"], weights["rand_r"])
    _ = ckks_degree2_activation(weights["x_poly"], 0.125, 0.5, 0.25)
    _ = ckks_degree4_activation(weights["x_poly"], -0.005, 0.2, 0.15)
    _ = zkml_fixed_point_linear(weights["zk_x"], weights["zk_w"], weights["zk_b"], 16.0)
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Combined Freivalds, CKKS, and ZKML latency: {lat_us:.2f} us")

In [ ]:
# Cryptographic Verification & Auditing
# 1. Freivalds: valid matmul residual is identically zero
norm_valid = torch.norm(v_valid).item()
norm_tampered = torch.norm(v_tamp).item()
print(f"Freivalds Residual: Valid Matmul = {norm_valid:.6e} vs Tampered Matmul = {norm_tampered:.4f}")
assert norm_valid < 1e-6, "Valid matrix multiplication must produce zero Freivalds residual!"
assert norm_tampered > 1.0, "Tampered matrix multiplication must produce non-zero Freivalds residual!"

# 2. CKKS Polynomial Approximation vs ReLU
relu_true = torch.relu(weights["x_poly"])
err_p2 = torch.mean(torch.abs(p2 - relu_true)).item()
err_p4 = torch.mean(torch.abs(p4 - relu_true)).item()
print(f"Polynomial Activation Approximation Error: Degree-2 = {err_p2:.4f}, Degree-4 = {err_p4:.4f}")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Freivalds' Algorithm** verifies dense matrix multiplications in $O(n^2)$ time with error $\le 2^{-k}$, bypassing cubic verifier bottlenecks.
2. **ZKML Fixed-Point** arithmetic maps real-valued weights to finite field elements $\mathbb{F}_p$ for non-interactive zk-SNARK proofs.
3. **CKKS Homomorphic Activations** replace non-algebraic ReLUs with low-degree polynomial approximations for encrypted inference.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [10_cryptographic_verifiable_ai_zkml_freivalds_and_homomorphic_ckks.md](../../markdown/11_ai_safety_privacy_and_verification/10_cryptographic_verifiable_ai_zkml_freivalds_and_homomorphic_ckks.md)
- **PyTorch 2.x Module:** [10_cryptographic_verifiable_ai_zkml_freivalds_and_homomorphic_ckks.py](../../pytorch2/11_ai_safety_privacy_and_verification/10_cryptographic_verifiable_ai_zkml_freivalds_and_homomorphic_ckks.py)
- **Functional JAX Module:** [10_cryptographic_verifiable_ai_zkml_freivalds_and_homomorphic_ckks.py](../../jax/11_ai_safety_privacy_and_verification/10_cryptographic_verifiable_ai_zkml_freivalds_and_homomorphic_ckks.py)
- **Native Rust Candle Track:** [10_cryptographic_verifiable_ai_zkml_freivalds_and_homomorphic_ckks.rs](../../candle/11_ai_safety_privacy_and_verification/10_cryptographic_verifiable_ai_zkml_freivalds_and_homomorphic_ckks.rs)
- **Hardware Kernel (CPU / WGSL):** [10_cryptographic_verifiable_ai_zkml_freivalds_and_homomorphic_ckks.rs](../../kernels/11_ai_safety_privacy_and_verification/10_cryptographic_verifiable_ai_zkml_freivalds_and_homomorphic_ckks.rs)
- **Golden SafeTensors:** [10_cryptographic_verifiable_ai_zkml_freivalds_and_homomorphic_ckks.safetensors](../../weights/11_ai_safety_privacy_and_verification/10_cryptographic_verifiable_ai_zkml_freivalds_and_homomorphic_ckks.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 09. OOD Detection Abstention Conformal and Algorithmic Fairness](09_ood_detection_abstention_conformal_and_algorithmic_fairness.ipynb) | [Master Index](../../README.md#11_ai_safety_privacy_and_verification) | [Next: 01. Torch Compile Modes Backends and Graph Breaks](../12_compilation_and_performance/01_torch_compile_modes_backends_and_graph_breaks.ipynb) &rarr;